# EDA: Catastro Hermosillo 2025

Este notebook analiza el dataset catastral, lo valida contra el PDF oficial de planos de valores,
lo cruza con el DENUE para asignar AGEBs y genera `data/processed/catastro_hermosillo.csv`.

**Fuentes:**
- `data/raw/catastro_hermosillo_2025.csv` — Tabla de valores por colonia/zona  
- `references/planos-valores-unitarios-catastro 2025.pdf` — Documento oficial (escaneado)  
- `data/raw/denue_sonora_2020.csv` — Crosswalk colonia → CVE_AGEB  
- `data/processed/train_ready.parquet` — Para analizar cobertura y correlaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from difflib import get_close_matches

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')

ROOT = Path('..')

## 1. Carga y limpieza del CSV catastral

In [ ]:
cat_raw = pd.read_csv(ROOT / 'data/raw/catastro_hermosillo_2025.csv', encoding='latin-1')
print('Columnas originales:', list(cat_raw.columns))
print('Shape:', cat_raw.shape)
cat_raw.head(8)

In [ ]:
# Renombrar columnas para trabajar más fácil
cat = cat_raw.copy()
cat.columns = ['ZONA', 'COLONIA', 'VALOR_STR']

# Parsear VALOR: quitar '$' y ',' → float
cat['VALOR'] = (
    cat['VALOR_STR']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
cat['VALOR'] = pd.to_numeric(cat['VALOR'], errors='coerce')
cat = cat.drop(columns='VALOR_STR')

# Normalizar COLONIA
cat['COLONIA'] = cat['COLONIA'].str.strip().str.upper()
cat['ZONA'] = cat['ZONA'].str.strip().str.upper()

print(f"Filas totales:       {len(cat)}")
print(f"Colonias únicas:     {cat['COLONIA'].nunique()}")
print(f"Zonas únicas:        {cat['ZONA'].nunique()}")
print(f"NaN en VALOR:        {cat['VALOR'].isna().sum()}")
cat.head(8)

## 2. Estadísticas del Valor Catastral

In [ ]:
print('=== Estadísticas del Valor Catastral Unitario ($/m²) ===')
print(cat['VALOR'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat['VALOR'].dropna().hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución del Valor Catastral ($/m²)')
axes[0].set_xlabel('$/m²')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(cat['VALOR'].median(), color='red', linestyle='--', label=f"Mediana: ${cat['VALOR'].median():,.0f}")
axes[0].legend()

np.log1p(cat['VALOR'].dropna()).hist(bins=60, ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Distribución en escala logarítmica')
axes[1].set_xlabel('log(VALOR + 1)')
axes[1].set_ylabel('Frecuencia')

plt.suptitle('Catastro Hermosillo 2025 — Valor Unitario ($/m²)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print('=== TOP 20 colonias más caras ===')
print(cat.nlargest(20, 'VALOR')[['ZONA', 'COLONIA', 'VALOR']].to_string(index=False))
print()
print('=== BOTTOM 20 colonias más baratas ===')
print(cat.nsmallest(20, 'VALOR')[['ZONA', 'COLONIA', 'VALOR']].to_string(index=False))

In [ ]:
zona_stats = (
    cat.groupby('ZONA')['VALOR']
    .agg(media='mean', maximo='max', minimo='min', n_colonias='count')
    .sort_values('media', ascending=False)
    .reset_index()
)
print('Top 20 zonas por valor promedio:')
print(zona_stats.head(20).round(0).to_string(index=False))

## 3. Validación contra el PDF oficial

El archivo `references/planos-valores-unitarios-catastro 2025.pdf` es un **PDF escaneado de 32 MB** 
(imágenes, sin capa de texto extraíble), por lo que no es posible compararlo programáticamente.

### ¿Qué es ese documento?
Es el acuerdo oficial del Cabildo de Hermosillo que fija los valores unitarios por zona catastral 
para el ejercicio fiscal 2025. Contiene:
- Mapas de polígonos de zonas (parte visual)
- Tabla de valores por zona/colonia (parte tabular — es lo que tenemos en el CSV)

### Cómo verificar manualmente
Abre el PDF y busca algunas de las zonas de la tabla siguiente para confirmar que los valores coinciden:

In [ ]:
# Tabla de referencia para comparación manual con el PDF
tabla_ref = (
    cat.groupby('ZONA')
    .agg(
        valor_min=('VALOR', 'min'),
        valor_max=('VALOR', 'max'),
        valor_medio=('VALOR', 'mean'),
        n_colonias=('COLONIA', 'count')
    )
    .reset_index()
    .sort_values('ZONA')
)
print('Tabla ZONA → VALORES (cotejar con el PDF):')
print(tabla_ref.to_string(index=False))

In [ ]:
# Colonias de referencia para spot-check manual con el PDF
# Estas son zonas con valores extremos o muy conocidas en Hermosillo
spot_check = cat[cat['COLONIA'].isin([
    'AMBAR', 'PARKVIEW', 'COUNTRY CLUB', 'CENTRO',
    'UNIVERSIDAD DE SONORA', 'VILLA DE SERIS', 'PERISUR'
])].sort_values('VALOR', ascending=False)
print('Colonias de referencia para spot-check:')
print(spot_check[['ZONA', 'COLONIA', 'VALOR']].to_string(index=False))

## 4. Cruce con DENUE → Asignación de CVE_AGEB

El `Mapa.json` solo tiene polígonos a nivel **municipio** (72 municipios de Sonora), no AGEBs.
En cambio, el DENUE de INEGI tiene la columna **'Área geoestadística básica'** con el código 
AGEB de cada negocio, junto con el nombre de la colonia.

Esto nos permite construir el crosswalk: **COLONIA → CVE_AGEB** directamente del DENUE, 
sin necesidad de geocoding ni descarga de polígonos externos.

In [ ]:
COLS_DENUE = [
    'Clave entidad', 'Clave municipio', 'Clave localidad',
    'Área geoestadística básica ', 'Nombre de asentamiento humano'
]

denue_raw = pd.read_csv(
    ROOT / 'data/raw/denue_sonora_2020.csv',
    encoding='latin-1',
    usecols=COLS_DENUE
)

hmo = denue_raw[
    (denue_raw['Clave municipio'] == 30) &
    (denue_raw['Clave entidad'] == 26)
].copy()

print(f"Registros DENUE Hermosillo: {len(hmo)}")
print(f"AGEBs únicos (DENUE):       {hmo['Área geoestadística básica '].nunique()}")
print(f"Colonias únicas (DENUE):    {hmo['Nombre de asentamiento humano'].nunique()}")

In [ ]:
# Construir CVE_AGEB completo (13 dígitos): ENT(2) + MUN(3) + LOC(4) + AGEB(4)
hmo['CVE_AGEB'] = (
    hmo['Clave entidad'].astype(str).str.zfill(2) +
    hmo['Clave municipio'].astype(str).str.zfill(3) +
    hmo['Clave localidad'].astype(str).str.zfill(4) +
    hmo['Área geoestadística básica '].astype(str).str.zfill(4)
)
hmo['COLONIA_DENUE'] = hmo['Nombre de asentamiento humano'].str.strip().str.upper()

print('Ejemplo CVE_AGEB construido:')
print(hmo[['COLONIA_DENUE', 'CVE_AGEB']].drop_duplicates().head(10).to_string(index=False))

In [ ]:
# Crosswalk: {COLONIA → lista de CVE_AGEBs}
# Una colonia puede tocar varios AGEBs (colonia grande o en límite entre AGEBs)
crosswalk = (
    hmo.groupby('COLONIA_DENUE')['CVE_AGEB']
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index()
    .rename(columns={'CVE_AGEB': 'AGEBS'})
)

print(f"Colonias en crosswalk DENUE: {len(crosswalk)}")
# Colonias que cruzan más de 1 AGEB
multi = crosswalk[crosswalk['AGEBS'].apply(len) > 1]
print(f"Colonias en más de 1 AGEB:   {len(multi)}")
crosswalk.head(10)

## 5. Match: Catastro ↔ DENUE

In [ ]:
denue_colonias = set(crosswalk['COLONIA_DENUE'])
cat['MATCH_EXACTO'] = cat['COLONIA'].isin(denue_colonias)

n_exact = cat['MATCH_EXACTO'].sum()
n_total = len(cat)
print(f"Colonias en catastro:         {n_total}")
print(f"Match exacto con DENUE:       {n_exact} ({n_exact/n_total*100:.1f}%)")
print(f"Sin match (van a fuzzy):      {n_total - n_exact} ({(n_total - n_exact)/n_total*100:.1f}%)")

In [ ]:
# Fuzzy matching para colonias sin match exacto
sin_match = cat[~cat['MATCH_EXACTO']].copy()
denue_lista = list(denue_colonias)

fuzzy_map = {}
for colonia in sin_match['COLONIA'].unique():
    matches = get_close_matches(colonia, denue_lista, n=1, cutoff=0.80)
    fuzzy_map[colonia] = matches[0] if matches else None

sin_match['COLONIA_FUZZY'] = sin_match['COLONIA'].map(fuzzy_map)

n_fuzzy = sin_match['COLONIA_FUZZY'].notna().sum()
n_sin_ageb = sin_match['COLONIA_FUZZY'].isna().sum()
print(f"Recuperadas por fuzzy match:  {n_fuzzy} ({n_fuzzy/(n_total - n_exact)*100:.1f}% de las sin match)")
print(f"Sin AGEB (quedarán fuera):    {n_sin_ageb}")
print()
print('Ejemplos de match fuzzy:')
print(sin_match[sin_match['COLONIA_FUZZY'].notna()][['COLONIA', 'COLONIA_FUZZY', 'VALOR']].head(20).to_string(index=False))

In [ ]:
# Colonias del catastro que no tienen AGEB (ni exacto ni fuzzy)
sin_ageb = sin_match[sin_match['COLONIA_FUZZY'].isna()][['ZONA', 'COLONIA', 'VALOR']]
print(f'Colonias sin asignación de AGEB ({len(sin_ageb)} registros):')
print(sin_ageb.to_string(index=False))

## 6. Dataset expandido con CVE_AGEB

In [ ]:
# Expandir colonias con match exacto
exact_df = cat[cat['MATCH_EXACTO']].merge(
    crosswalk, left_on='COLONIA', right_on='COLONIA_DENUE', how='left'
).explode('AGEBS').rename(columns={'AGEBS': 'CVE_AGEB'})[['CVE_AGEB', 'ZONA', 'COLONIA', 'VALOR']]

# Expandir colonias con match fuzzy
fuzzy_df_src = sin_match[sin_match['COLONIA_FUZZY'].notna()].copy()
fuzzy_df = fuzzy_df_src.merge(
    crosswalk, left_on='COLONIA_FUZZY', right_on='COLONIA_DENUE', how='left'
).explode('AGEBS').rename(columns={'AGEBS': 'CVE_AGEB'})[['CVE_AGEB', 'ZONA', 'COLONIA', 'VALOR']]

catastro_ageb = (
    pd.concat([exact_df, fuzzy_df], ignore_index=True)
    .drop_duplicates()
    .dropna(subset=['CVE_AGEB'])
    .sort_values(['CVE_AGEB', 'VALOR'], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Filas en catastro_ageb: {len(catastro_ageb)}")
print(f"AGEBs cubiertos:        {catastro_ageb['CVE_AGEB'].nunique()}")
print(f"Colonias con AGEB:      {catastro_ageb['COLONIA'].nunique()}")
catastro_ageb.head(10)

## 7. Análisis de cobertura vs train_ready

In [ ]:
train = pd.read_parquet(ROOT / 'data/processed/train_ready.parquet')

agebs_train    = set(train.index)
agebs_catastro = set(catastro_ageb['CVE_AGEB'])

cubiertos  = agebs_train & agebs_catastro
sin_cubrir = agebs_train - agebs_catastro

print(f"AGEBs en train_ready:            {len(agebs_train)}")
print(f"AGEBs con dato catastral:        {len(cubiertos)} ({len(cubiertos)/len(agebs_train)*100:.1f}%)")
print(f"AGEBs sin dato catastral (→ 0):  {len(sin_cubrir)} ({len(sin_cubrir)/len(agebs_train)*100:.1f}%)")
print()
print(f"AGEBs en catastro FUERA de train_ready: {len(agebs_catastro - agebs_train)}")

In [ ]:
# Calcular VALOR_CATASTRAL_MAX por AGEB y unir a train_ready
valor_max = catastro_ageb.groupby('CVE_AGEB')['VALOR'].max().rename('VALOR_CATASTRAL_MAX')

train_con_catastro = train.join(valor_max, how='left')
train_con_catastro['VALOR_CATASTRAL_MAX'] = train_con_catastro['VALOR_CATASTRAL_MAX'].fillna(0)

print('VALOR_CATASTRAL_MAX por AGEB (incluyendo ceros):')
print(train_con_catastro['VALOR_CATASTRAL_MAX'].describe().round(0))
print(f"\nAGEBs con VALOR > 0: {(train_con_catastro['VALOR_CATASTRAL_MAX'] > 0).sum()}")
print(f"AGEBs con VALOR = 0: {(train_con_catastro['VALOR_CATASTRAL_MAX'] == 0).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribución de VALOR_CATASTRAL_MAX (solo AGEBs con dato)
con_dato = train_con_catastro[train_con_catastro['VALOR_CATASTRAL_MAX'] > 0]
con_dato['VALOR_CATASTRAL_MAX'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('VALOR_CATASTRAL_MAX por AGEB\n(AGEBs con dato catastral)')
axes[0].set_xlabel('$/m²')

# Boxplot por clase
train_con_catastro.boxplot(column='VALOR_CATASTRAL_MAX', by='abandono_alto', ax=axes[1])
axes[1].set_title('VALOR_CATASTRAL_MAX vs abandono_alto')
axes[1].set_xlabel('0 = estable, 1 = alto riesgo de abandono')
axes[1].set_ylabel('$/m²')
plt.sca(axes[1])
plt.title('VALOR_CATASTRAL_MAX vs Clase')

# Scatter: VALOR_CATASTRAL_MAX vs SCORE_REZAGO
clases = train_con_catastro['abandono_alto']
scatter = axes[2].scatter(
    train_con_catastro['VALOR_CATASTRAL_MAX'],
    train_con_catastro['SCORE_REZAGO'],
    c=clases, cmap='RdYlGn_r', alpha=0.6, s=20
)
plt.colorbar(scatter, ax=axes[2], label='abandono_alto')
axes[2].set_title('Valor Catastral vs Rezago Habitacional')
axes[2].set_xlabel('VALOR_CATASTRAL_MAX ($/m²)')
axes[2].set_ylabel('SCORE_REZAGO')

plt.suptitle('Análisis de VALOR_CATASTRAL_MAX', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 8. Correlaciones con el target y otras features

In [ ]:
print('=== VALOR_CATASTRAL_MAX por clase de abandono ===')
print(train_con_catastro.groupby('abandono_alto')['VALOR_CATASTRAL_MAX'].describe().round(0))
print()

features_corr = [
    'VALOR_CATASTRAL_MAX', 'SCORE_REZAGO', 'abandono_alto',
    'n_inmobiliarias', 'VPH_AUTOM', 'GRAPROES', 'HACINAMIENTO'
]
# Solo las columnas que existen en train_con_catastro
cols_disponibles = [c for c in features_corr if c in train_con_catastro.columns]

corr = train_con_catastro[cols_disponibles].corr()
print('=== Correlaciones con VALOR_CATASTRAL_MAX ===')
print(corr['VALOR_CATASTRAL_MAX'].sort_values().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
subset_corr = train_con_catastro[cols_disponibles].corr()
mask = np.triu(np.ones_like(subset_corr, dtype=bool))
sns.heatmap(
    subset_corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax, square=True
)
ax.set_title('Correlaciones — Features seleccionadas + VALOR_CATASTRAL_MAX')
plt.tight_layout()
plt.show()

## 9. Guardar `data/processed/catastro_hermosillo.csv`

Este CSV contiene una fila por cada par (colonia, AGEB). El feature `VALOR_CATASTRAL_MAX` 
se calculará en el script de feature engineering (máximo por AGEB).

In [ ]:
OUTPUT = ROOT / 'data/processed/catastro_hermosillo.csv'
catastro_ageb.to_csv(OUTPUT, index=False)

print(f'Guardado en: {OUTPUT}')
print(f'Shape: {catastro_ageb.shape}')
print(f'Columnas: {list(catastro_ageb.columns)}')
print()
catastro_ageb.head(10)

## 10. Conclusión: ¿Vale la pena como feature?

Revisa los resultados de las celdas anteriores y agrega aquí tus observaciones.

**Criterios para decidir si integrar la feature:**
- **Cobertura:** ¿Qué % de los 665 AGEBs tienen dato catastral? Si es < 30%, el feature tiene poco poder.
- **Discriminación:** ¿La clase 0 (estable) tiene mayor VALOR_CATASTRAL_MAX que la clase 1 (abandono)?  
  Si sí → el feature captura el punto ciego de colonias de lujo.
- **Correlación con abandono_alto:** Se espera negativa (más caro → menos abandono).
- **Independencia con otras features:** ¿Aporta información que n_inmobiliarias o SCORE_REZAGO no tienen?

**Siguiente paso si el análisis es favorable:**  
Ejecutar `src/features/build_catastro.py` y `src/features/build_features.py` para integrar 
`VALOR_CATASTRAL_MAX` en `train_ready.parquet` y usarla en los modelos.